# Лабораторная работа 3 - IEnv | Обработка датасета

## 1 - Подключение Spark и PySpark

In [21]:
from pyspark.sql import SparkSession

# создаем сессию с более старым парсером дат для совместимости
# и отдельно докачиваем пакет для сохранения в avro
spark = SparkSession.builder \
    .appName("Chicago_lab") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.1.1") \
    .getOrCreate()

print("\nsession was successfully created")


session was successfully created


## 2 - Загрузка датасетов из скачанных csv файлов

In [22]:
# header - берем описание столбцов из первой строки (названия)
# inferSchema - самостоятельно определяем типы данных по столбцам
taxi_df = spark.read.csv("chicago_taxi_trips.csv", header=True, inferSchema=True)
weather_df = spark.read.csv("chicago_weather.csv", header=True, inferSchema=True)

print("\ndatasets are loaded from csv files")


datasets are loaded from csv files


## 3 - Анализ по chicago_taxi_trips

In [23]:
from pyspark.sql.functions import col, count, when

print(f"Общее количество строк: {taxi_df.count()}") # количество записей по датасету
print(f"Схема данных:")
taxi_df.printSchema()
print(f"Название колонок:")
columns = taxi_df.columns
# форматируем вывод по 5 названий колонок
for i in range(0, len(columns), 5):
    column_batch = columns[i:i+5]
    print(i + 1, "-", i + 5, ": ", " || ".join(column_batch))

Общее количество строк: 150000
Схема данных:
root
 |-- :id: string (nullable = true)
 |-- :version: string (nullable = true)
 |-- :created_at: timestamp (nullable = true)
 |-- :updated_at: timestamp (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: string (nullable = true)
 |-- trip_end_timestamp: string (nullable = true)
 |-- trip_seconds: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_census_tract: long (nullable = true)
 |-- dropoff_census_tract: long (nullable = true)
 |-- pickup_community_area: integer (nullable = true)
 |-- dropoff_community_area: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- company: string (nullable = true)
 |-- pickup_centroid_latitude: doubl

In [24]:
def check_missing_data(df, num_rows=1000, only_total = False):
    limited_df = df.limit(num_rows) # ограничение по количеству строк
    
    # команда для счета NULL значений по каждой из колонок
    # alias - для адекватного названия колонок вместо SQL команд
    command = [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
    result = limited_df.agg(*command)
    
    if not only_total:
        # применяем команды к нашему обрезанному датасету
        print(f"Количество NULL в первых {num_rows} строках:")
        # отображаем результат по колонкам    
        result.show(vertical=True)

    print(f"Общее количество значений NULL: {sum(list(result.collect()[0]))}")

check_missing_data(taxi_df) # подробный анализ первых 1к строк
check_missing_data(taxi_df, 2000, True) # только общее количество NULL по первым 2к строк

Количество NULL в первых 1000 строках:
-RECORD 0-------------------------
 :id                        | 0   
 :version                   | 0   
 :created_at                | 0   
 :updated_at                | 0   
 trip_id                    | 0   
 taxi_id                    | 0   
 trip_start_timestamp       | 0   
 trip_end_timestamp         | 0   
 trip_seconds               | 0   
 trip_miles                 | 0   
 pickup_census_tract        | 849 
 dropoff_census_tract       | 866 
 pickup_community_area      | 34  
 dropoff_community_area     | 122 
 fare                       | 1   
 tips                       | 1   
 tolls                      | 1   
 extras                     | 1   
 trip_total                 | 1   
 payment_type               | 0   
 company                    | 0   
 pickup_centroid_latitude   | 34  
 pickup_centroid_longitude  | 34  
 pickup_centroid_location   | 34  
 dropoff_centroid_latitude  | 119 
 dropoff_centroid_longitude | 119 
 dropoff_centroi

In [25]:
print("Список уникальных компаний:")
unique_companies = taxi_df.select("company").distinct()
unique_companies.show(truncate=False) # выключаем обрезку названий
print(f"\nОбщее количество: {unique_companies.count()}") # показываем также общее кол-во

Список уникальных компаний:
+------------------------------------+
|company                             |
+------------------------------------+
|3556 - 36214 RC Andrews Cab         |
|Chicago Taxicab                     |
|4053 - 40193 Adwar H. Nikola        |
|Taxi Affiliation Services           |
|Top Cab                             |
|5 Star Taxi                         |
|Metro Jet Taxi A.                   |
|U Taxicab                           |
|Flash Cab                           |
|Choice Taxi Association             |
|3591 - 63480 Chuks Cab              |
|5167 - 71969 5167 Taxi Inc          |
|Chicago City Taxi Association       |
|Setare Inc                          |
|Sun Taxi                            |
|Blue Ribbon Taxi Association        |
|Patriot Taxi Dba Peace Taxi Associat|
|6574 - Babylon Express Inc.         |
|Petani Cab Corp                     |
|Medallion Leasin                    |
+------------------------------------+
only showing top 20 rows

Общее коли

In [26]:
unique_taxis = taxi_df.select("taxi_id").distinct().count()
print(f"Уникальных машин такси работает в Чикаго: {unique_taxis}")

Уникальных машин такси работает в Чикаго: 2320


## 4 - chicago_weather анализ

In [27]:
print(f"Общее количество строк: {weather_df.count()}") # количество записей по датасету
print(f"Схема данных:")
weather_df.printSchema()
print(f"Название колонок:")
columns = weather_df.columns
# форматируем вывод по 5 названий колонок
for i in range(0, len(columns), 5):
    column_batch = columns[i:i+5]
    print(i + 1, "-", i + 5, ": ", " || ".join(column_batch))

Общее количество строк: 14
Схема данных:
root
 |-- STATION: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- AWND: double (nullable = true)
 |-- PRCP: double (nullable = true)
 |-- SNOW: double (nullable = true)
 |-- SNWD: double (nullable = true)
 |-- TAVG: string (nullable = true)
 |-- TMAX: double (nullable = true)
 |-- TMIN: double (nullable = true)
 |-- WT01: double (nullable = true)
 |-- WT02: double (nullable = true)
 |-- WT03: string (nullable = true)
 |-- WT04: double (nullable = true)
 |-- WT05: double (nullable = true)
 |-- WT06: double (nullable = true)
 |-- WT07: string (nullable = true)
 |-- WT08: string (nullable = true)
 |-- WT09: string (nullable = true)

Название колонок:
1 - 5 :  STATION || DATE || AWND || PRCP || SNOW
6 - 10 :  SNWD || TAVG || TMAX || TMIN || WT01
11 - 15 :  WT02 || WT03 || WT04 || WT05 || WT06
16 - 20 :  WT07 || WT08 || WT09


In [28]:
check_missing_data(weather_df)

Количество NULL в первых 1000 строках:
-RECORD 0------
 STATION | 0   
 DATE    | 0   
 AWND    | 0   
 PRCP    | 0   
 SNOW    | 0   
 SNWD    | 0   
 TAVG    | 14  
 TMAX    | 0   
 TMIN    | 0   
 WT01    | 4   
 WT02    | 11  
 WT03    | 14  
 WT04    | 13  
 WT05    | 13  
 WT06    | 12  
 WT07    | 14  
 WT08    | 14  
 WT09    | 14  

Общее количество значений NULL: 123


## 5 - chicago_taxi_trips ETL обработка

### 5.1 - получение временных данных

In [29]:
# ETL - extract, transform, load

# временные диапазоны датасетов
# taxi - 21 января 2024 - 01 февраля 2024
# weather - 20 января 2024 - 02 февраля 2024
from pyspark.sql.functions import col, year, month, dayofmonth, dayofyear, hour, minute, weekofyear, unix_timestamp, udf, round as spark_round
from pyspark.sql.types import DoubleType

# в полученном датасете дата стандартизирована
# ex: 2024-01-31T23:45:00.000
# поэтому мы просто преобразуем значение к timestamp 
# и помещаем оттуда необходимые значения в новый датасет
time_features_df = taxi_df.select(
    col("trip_id").alias("trip_id"),
    year(col("trip_start_timestamp").cast("timestamp")).alias("year"),
    month(col("trip_start_timestamp").cast("timestamp")).alias("month"),
    dayofmonth(col("trip_start_timestamp").cast("timestamp")).alias("day"),
    dayofyear(col("trip_start_timestamp").cast("timestamp")).alias("dayofyear"),
    hour(col("trip_start_timestamp").cast("timestamp")).alias("hour"),
    minute(col("trip_start_timestamp").cast("timestamp")).alias("min"),
    weekofyear(col("trip_start_timestamp").cast("timestamp")).alias("week_no"),
    unix_timestamp(col("trip_start_timestamp").cast("timestamp")).alias("unix_ts")
)

# соединяем таблицы по айди поездки
taxi_df = taxi_df.join(time_features_df, "trip_id", "inner")

# показываем первые 10 строк по новым колонкам из таблицы
taxi_df.select("trip_id", "year", "month", "dayofyear", "unix_ts").show(10, truncate=False)

+----------------------------------------+----+-----+---------+----------+
|trip_id                                 |year|month|dayofyear|unix_ts   |
+----------------------------------------+----+-----+---------+----------+
|763a7eb767f8fa5aa05876cd5863c50ba941a646|2024|1    |31       |1706733900|
|ef73027c67c779453d73ab0ba04c0ec1f0ec362d|2024|1    |31       |1706733900|
|281e12c3792a7ac076fdad68d8733ca30c3d3e69|2024|1    |31       |1706733900|
|56d1c8214e3dadf4bc1dc033054cbfdf037a427f|2024|1    |31       |1706733900|
|4e52a1f0e67f3f421217dd0ead295394c72295f9|2024|1    |31       |1706733900|
|f80c5789c21340336a69dff05a6771445a625f2b|2024|1    |31       |1706733900|
|08abaa6ac1b09bcb5e574572d0a672b4a323b6b5|2024|1    |31       |1706733900|
|ad635c70c1a3dd92ebf86654d5bc216c3e99ddad|2024|1    |31       |1706733900|
|b529bc86153d922a222db29ade97a38a95609b7e|2024|1    |31       |1706733900|
|331bdf95aa0fc4cc6b63e89033af555ebecd93c9|2024|1    |31       |1706733900|
+------------------------

### 5.2 - функция очистки формата денежных колонок

In [30]:
def clean_money(value):
    if value is None:
        return 0.0
    
    # меняем ненужные знаки, удаляем незначащие пробелы
    formated_str = str(value).replace("$", "").replace(",",".").strip()

    try:
        return float(formated_str)
    except ValueError:
        return 0.0
    
# демонстрация возможности собственной функции обработки, 
# если встроенных не достаточно
money_udf = udf(clean_money, DoubleType())
money_columns = ["fare", "tips", "tolls", "extras", "trip_total"]

for column in money_columns:
    taxi_df = taxi_df.withColumn(column, money_udf(col(column)))

print("Измененные денежные поля:")
taxi_df.select(money_columns).show(10)


Измененные денежные поля:
+-----+-----+-----+------+----------+
| fare| tips|tolls|extras|trip_total|
+-----+-----+-----+------+----------+
|34.25|  0.0|  0.0|   0.0|     34.25|
| 23.0|  0.0|  0.0|   0.0|      23.0|
| 57.0| 18.4|  0.0|   4.0|      79.4|
|43.75| 9.65|  0.0|   4.0|      57.9|
|  7.0|  4.0|  0.0|   1.5|      13.0|
| 5.25|  0.0|  0.0|   0.0|      5.25|
| 45.0| 7.58|  0.0|   5.0|     58.08|
|46.25|10.25|  0.0|   4.5|      61.5|
| 10.5| 11.2|  0.0|  45.0|      66.7|
|37.75| 4.32|  0.0|   5.0|     47.57|
+-----+-----+-----+------+----------+
only showing top 10 rows


### 5.3 - число поездок за дату

In [31]:
trips_2019 = taxi_df.filter(
    (col("year") == 2019) & 
    (col("month") == 6) & 
    (col("day") == 14)
).count()

print(f"Количество поездок за 14.06.2019: {trips_2019}")

trip_day = 25
trip_month = 1
trip_year = 2024

trips_by_date = taxi_df.filter(
    (col("year") == trip_year) & 
    (col("month") == trip_month) & 
    (col("day") == trip_day)
).count()

print(f"Количество поездок за {trip_day:02d}.{trip_month:02d}.{trip_year:04d}: {trips_by_date}")

Количество поездок за 14.06.2019: 0
Количество поездок за 25.01.2024: 18267


### 5.4 - места выдачи такси в каждой доне

In [32]:
# датасет по местам выдачи такси
# округляем до 4 знаков после запятой с помощью spark_round 
# отбрасываем пустые зоны с помощью isNotNull 
# берем только уникальные
community_areas_df = taxi_df.select(
    col("pickup_community_area").alias("community_area").cast("integer"),
    spark_round(col("pickup_centroid_latitude"), 4).alias("centroid_latitude"),
    spark_round(col("pickup_centroid_longitude"), 4).alias("centroid_longitude")
).filter(col("community_area").isNotNull()).distinct()

print("Места выдачи такси:")
community_areas_df.show(10)

Места выдачи такси:
+--------------+-----------------+------------------+
|community_area|centroid_latitude|centroid_longitude|
+--------------+-----------------+------------------+
|             6|          41.9348|          -87.6399|
|            28|          41.8704|          -87.6751|
|            59|          41.8299|          -87.6725|
|            61|           41.809|          -87.6592|
|            38|          41.8129|          -87.6179|
|            77|          41.9867|          -87.6634|
|             8|          41.9075|          -87.6358|
|             8|          41.9028|          -87.6261|
|            44|          41.7402|           -87.616|
|            26|          41.8786|          -87.7302|
+--------------+-----------------+------------------+
only showing top 10 rows


### 5.5 - сводка данных для каждой поездки

In [33]:
# получаем итоговые сведения в соответствие с ожидаемым результатом
# итоговые имена колонок тоже из ожидаемых результатов
cabs_df = taxi_df.select(
    col("taxi_id").cast("string"),
    col("trip_id").cast("string"),
    col("trip_start_timestamp").cast("timestamp"),
    col("trip_end_timestamp").cast("timestamp"),
    col("trip_seconds").cast("integer"),
    col("trip_miles").cast("double"),
    col("pickup_community_area").cast("integer"),
    col("dropoff_community_area").cast("integer"),
    col("fare").cast("double"),
    col("tips").alias("tip").cast("double"), # в результате int, хотя должно быть double
    col("tolls").alias("additional_charges").cast("double"),
    col("extras").alias("extra").cast("double"),
    col("trip_total").cast("double"),
    col("payment_type").cast("string"),
    col("company").alias("taxi_company").cast("string")
)

cabs_df.show(10)
cabs_df.printSchema()

+--------------------+--------------------+--------------------+-------------------+------------+----------+---------------------+----------------------+-----+-----+------------------+-----+----------+------------+--------------------+
|             taxi_id|             trip_id|trip_start_timestamp| trip_end_timestamp|trip_seconds|trip_miles|pickup_community_area|dropoff_community_area| fare|  tip|additional_charges|extra|trip_total|payment_type|        taxi_company|
+--------------------+--------------------+--------------------+-------------------+------------+----------+---------------------+----------------------+-----+-----+------------------+-----+----------+------------+--------------------+
|a500c0da8fa893c9b...|763a7eb767f8fa5aa...| 2024-01-31 23:45:00|2024-02-01 00:30:00|        2640|       0.6|                   13|                  NULL|34.25|  0.0|               0.0|  0.0|     34.25|     Unknown|Taxi Affiliation ...|
|6278c1674ea7d215c...|ef73027c67c779453...| 2024-01-31 2

## 6 - ETL обработка chicago_weather

### 6.1 - извлечение временных данных

In [34]:
# достаем нужные данные из полей
weather_dates_df = weather_df.select(
    col("DATE").alias("DATE"),
    year(col("DATE")).alias("year"),
    month(col("DATE")).alias("month"),
    dayofyear(col("DATE")).alias("dayofyear")
)

# соединяем таблицы по дате (уникальна)
weather_df = weather_df.join(weather_dates_df, "DATE", "inner")

# показываем первые 10 строк по новым колонкам из таблицы
weather_df.select("DATE", "year", "month", "dayofyear").show(10, truncate=False)

+----------+----+-----+---------+
|DATE      |year|month|dayofyear|
+----------+----+-----+---------+
|2024-01-20|2024|1    |20       |
|2024-01-21|2024|1    |21       |
|2024-01-22|2024|1    |22       |
|2024-01-23|2024|1    |23       |
|2024-01-24|2024|1    |24       |
|2024-01-25|2024|1    |25       |
|2024-01-26|2024|1    |26       |
|2024-01-27|2024|1    |27       |
|2024-01-28|2024|1    |28       |
|2024-01-29|2024|1    |29       |
+----------+----+-----+---------+
only showing top 10 rows


### 6.2 и 6.3 - выборки температуры и погодных условий

In [35]:
# необходимые поля для выборки температуры
temperature_df = weather_df.select(
    "DATE", "AWND", "PRCP", "SNOW", "SNWD", "TMAX", "TMIN", "TAVG", 
    "year", "dayofyear", "month"
)

# необходимые поля для выборки погодных условий
# с нормальными названиями из ожидаемых результатов
weather_conditions_df = weather_df.select(
    "DATE", "year", "dayofyear", "month",
    col("WT01").alias("FOG"),
    col("WT02").alias("HEAVY_FOG"),
    col("WT03").alias("THNDR"),
    col("WT04").alias("ICE"),
    col("WT05").alias("HAIL"),
    col("WT09").alias("HVSNOW")
)

temperature_df.show(5)
weather_conditions_df.show(5)

+----------+----+----+----+-----+----+-----+----+----+---------+-----+
|      DATE|AWND|PRCP|SNOW| SNWD|TMAX| TMIN|TAVG|year|dayofyear|month|
+----------+----+----+----+-----+----+-----+----+----+---------+-----+
|2024-01-20| 4.4| 0.0| 0.0|100.0|-7.7|-15.5|NULL|2024|       20|    1|
|2024-01-21| 4.4| 0.0| 0.0|100.0|-6.0|-17.1|NULL|2024|       21|    1|
|2024-01-22| 5.1| 0.8| 0.0|100.0| 1.1| -6.0|NULL|2024|       22|    1|
|2024-01-23| 1.7| 9.1| 0.0|100.0| 2.8|  0.6|NULL|2024|       23|    1|
|2024-01-24| 2.6| 4.6| 0.0| 80.0| 3.3|  1.1|NULL|2024|       24|    1|
+----------+----+----+----+-----+----+-----+----+----+---------+-----+
only showing top 5 rows
+----------+----+---------+-----+----+---------+-----+----+----+------+
|      DATE|year|dayofyear|month| FOG|HEAVY_FOG|THNDR| ICE|HAIL|HVSNOW|
+----------+----+---------+-----+----+---------+-----+----+----+------+
|2024-01-20|2024|       20|    1|NULL|     NULL| NULL|NULL|NULL|  NULL|
|2024-01-21|2024|       21|    1|NULL|     NULL| 

## 7 - Расширенная таблица с фактами (поездки + температура)

### 7.2 - фильтрация нулевых данных

In [36]:
# меняем шаги 7.1 и 7.2 местами, чтобы сначала очистить данные, 
# а затем создать представление таблиц

# очищаем данные с NULL
taxi_filtered_df = taxi_df.filter(
    col("pickup_centroid_latitude").isNotNull() & 
    col("pickup_centroid_longitude").isNotNull() & 
    col("dropoff_centroid_latitude").isNotNull() & 
    col("dropoff_centroid_longitude").isNotNull()
)

taxi_count = taxi_df.count()
taxi_filtered_count = taxi_filtered_df.count()
if taxi_count > taxi_filtered_count:
    print("Отфильтрованная выборка имеет меньше записей, " +
          f"чем начальная: {taxi_count} и {taxi_filtered_count} записей")
else:
    print("Вероятно, в начальной выборке нет записей с NULL, " +
          "поэтому они одинакового размера с отфильтрованной." +
          f"{taxi_count} и {taxi_filtered_count} записей")

Отфильтрованная выборка имеет меньше записей, чем начальная: 150000 и 136245 записей


### 7.1 - временные представления таблиц

In [37]:
# создаем представления обоих таблиц
# для того чтобы была возможность обращаться к ним
# в запросах через spark SQL
# иначе ядро не увидит таблицы
# т.е. иммитация существования таблицы в БД
taxi_filtered_df.createOrReplaceTempView("taxi_trips_view")
temperature_df.createOrReplaceTempView("temperature_view")


### 7.3 - объединение таблицы такси с температурой

In [38]:
# получаем новую таблицу через spark sql
# с помощью объединения по полям дат
# выбираем все поля из такси и необходимые поля из температуры
extended_taxi_trips = spark.sql("""
    SELECT 
        taxi.*,
        weather.AWND,
        weather.PRCP,
        weather.SNOW,
        weather.SNWD,
        weather.TMAX,
        weather.TMIN,
        weather.TAVG
    FROM taxi_trips_view taxi
    INNER JOIN temperature_view weather 
    ON taxi.year = weather.year AND taxi.dayofyear = weather.dayofyear
""")

# удаляем лишние служебные поля
columns_to_drop = [":id", ":version", ":created_at", ":updated_at"]
extended_taxi_trips = extended_taxi_trips.drop(*columns_to_drop)

# отображаем итоговую выборку
extended_taxi_trips.printSchema()
extended_taxi_trips.show(10)

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: string (nullable = true)
 |-- trip_end_timestamp: string (nullable = true)
 |-- trip_seconds: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_census_tract: long (nullable = true)
 |-- dropoff_census_tract: long (nullable = true)
 |-- pickup_community_area: integer (nullable = true)
 |-- dropoff_community_area: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- company: string (nullable = true)
 |-- pickup_centroid_latitude: double (nullable = true)
 |-- pickup_centroid_longitude: double (nullable = true)
 |-- pickup_centroid_location: string (nullable = true)
 |-- dropoff_centroid_latitude: double (nullable = true)
 |-- dropoff_centroid_

+--------------------+--------------------+--------------------+--------------------+------------+----------+-------------------+--------------------+---------------------+----------------------+-----+-----+-----+------+----------+------------+--------------------+------------------------+-------------------------+------------------------+-------------------------+--------------------------+-------------------------+----+-----+---+---------+----+---+-------+----------+----+----+----+-----+----+----+----+
|             trip_id|             taxi_id|trip_start_timestamp|  trip_end_timestamp|trip_seconds|trip_miles|pickup_census_tract|dropoff_census_tract|pickup_community_area|dropoff_community_area| fare| tips|tolls|extras|trip_total|payment_type|             company|pickup_centroid_latitude|pickup_centroid_longitude|pickup_centroid_location|dropoff_centroid_latitude|dropoff_centroid_longitude|dropoff_centroid_location|year|month|day|dayofyear|hour|min|week_no|   unix_ts|AWND|PRCP|SNOW| S

Traceback (most recent call last):
  File "/home/oleg/masters_study/computing_in_internet/spark-ienv/.venv/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/home/oleg/masters_study/computing_in_internet/spark-ienv/.venv/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe


## 8 - Сохранение в разных форматах данных

In [39]:
# avro - построчный формат, данные хранятся внутри файла в формате json
# эффективен в системах с активной записью, а так же если часто меняется структура таблицы
# для чтения по колонке приходится читать все данные

# parquet - колоночный формат данных, подходит для систем с активным чтением данных
# хорошо сжимает данные за счет однотипности данных в колонке
# при чтении по колонке делает это очень эффективно 
# (данные лежат в одном месте друг за другом)
# колоночный формат хуже для записи

# orc - также колоночный формат, имеет ещё более высокую степень сжатия данных 
# и сложные индексы внутри файла 
# по сравнению с parquet хуже поддерживает вложенные структуры

import os

all_dataframes_to_save = {
    "taxi_trips": extended_taxi_trips,
    "weather_conditions": weather_conditions_df,
    "temperature": temperature_df,
    "community_areas": community_areas_df,
    "cabs": cabs_df,
}

dir_to_save = "data"

for name, df in all_dataframes_to_save.items():
    # паркет и орк поддерживаются по умолчанию
    df.write.mode("overwrite").parquet(f"{dir_to_save}/parquet/{name}")
    df.write.mode("overwrite").orc(f"{dir_to_save}/orc/{name}")
    print(f"Успешная запись {name} в форматах parquet и orc")

    # для avro необходим дополнительный пакет org.apache.spark:spark-avro_2.12:4.1.1
    # который мы устанавливали при создании сессии
    try:
        df.write.mode("overwrite").format("avro").save(f"{dir_to_save}/avro/{name}")
        print(f"Успешная запись {name} в формат avro")
    except Exception as e:
        print("Необходимо установить соответствующий пакет для avro")

26/05/18 18:32:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Успешная запись taxi_trips в форматах parquet и orc


Успешная запись taxi_trips в формат avro
Успешная запись weather_conditions в форматах parquet и orc
Успешная запись weather_conditions в формат avro
Успешная запись temperature в форматах parquet и orc
Успешная запись temperature в формат avro
Успешная запись community_areas в форматах parquet и orc
Успешная запись community_areas в формат avro


26/05/18 18:32:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Успешная запись cabs в форматах parquet и orc


Успешная запись cabs в формат avro


In [40]:
# вывод всех файлов с их размером для сравнения
# самый емкий размер - orc
# самый большой - avro
!(cd data && du -sh */*)

29M	avro/cabs
12K	avro/community_areas
34M	avro/taxi_trips
12K	avro/temperature
12K	avro/weather_conditions
6,7M	orc/cabs
12K	orc/community_areas
8,4M	orc/taxi_trips
12K	orc/temperature
12K	orc/weather_conditions
11M	parquet/cabs
16K	parquet/community_areas
12M	parquet/taxi_trips
12K	parquet/temperature
12K	parquet/weather_conditions
